In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Load DefenseFinder
df_dir = "/home/ottena/mags_project/defensefinder_analysis/output"
df_files = glob.glob(os.path.join(df_dir, "*/*_defense_finder_systems.tsv"))

df_dfs = []
for f in df_files:
    genome = f.split("/")[-2]
    tmp = pd.read_csv(f, sep="\t")
    tmp["genome"] = genome
    df_dfs.append(tmp)

df_defense = pd.concat(df_dfs, ignore_index=True)
df_defense["tool"] = "DefenseFinder"

# Load PADLOC
pl_dir = "/home/ottena/mags_project/padloc_analysis/output"
pl_files = glob.glob(os.path.join(pl_dir, "*/*_padloc.csv"))

pl_dfs = []
for f in pl_files:
    genome = f.split("/")[-2]
    tmp = pd.read_csv(f)
    tmp["genome"] = genome
    pl_dfs.append(tmp)

df_padloc = pd.concat(pl_dfs, ignore_index=True)
df_padloc["tool"] = "PADLOC"

# Load labels
labels = pd.read_csv("/home/ottena/mags_project/pankmer_analysis/label_lookup_v3.csv")

# Merge labels
df_defense = df_defense.merge(labels, left_on="genome", right_on="original_name", how="left")
df_padloc = df_padloc.merge(labels, left_on="genome", right_on="original_name", how="left")

# Add groups
def get_group(label):
    if "NCBI" in str(label):
        return "NCBI Reference"
    elif "Masinjere" in str(label):
        return "Masinjere MAG"
    elif "Limera" in str(label):
        return "Limera MAG"
    return "Unknown"

df_defense["group"] = df_defense["plot_label"].apply(get_group)
df_padloc["group"] = df_padloc["plot_label"].apply(get_group)

print("DefenseFinder:", df_defense.shape)
print("PADLOC:", df_padloc.shape)

DefenseFinder: (682, 14)
PADLOC: (2734, 24)


In [ ]:
import subprocess
subprocess.run(["pip", "install", "matplotlib-venn"], check=True)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib_venn import venn2
import pandas as pd
import seaborn as sns
from numpy.polynomial.polynomial import polyfit

# Pastel color scheme
tool_colors = {"DefenseFinder": "#7BB8D4", "PADLOC": "#F4A6A0"}
group_colors = {"Masinjere MAG": "#FFBE86", "Limera MAG": "#A8C8E8", "NCBI Reference": "#A8D8A8"}
func_colors = {
    "Restriction-Modification": "#AEC6CF",
    "CRISPR-Cas": "#FFB7B2",
    "Abortive Infection": "#B5EAD7",
    "DNA Methylation & BREX": "#FFDAC1",
    "DNA Modification & Other": "#C7CEEA",
    "Toxin-Antitoxin": "#F2C6DE",
    "Innate-like / Other": "#D4D4D4",
}
genome_counts_map = {"NCBI Reference": 25, "Limera MAG": 16, "Masinjere MAG": 20}
groups_list = ["NCBI Reference", "Limera MAG", "Masinjere MAG"]
func_cats = list(func_colors.keys())

# --- Plot 1: Normalized bar ---
df_counts = df_defense.groupby("group").size().reset_index(name="DefenseFinder")
pl_counts = df_padloc.groupby("group").size().reset_index(name="PADLOC")
combined = df_counts.merge(pl_counts, on="group")
combined["DefenseFinder_norm"] = combined.apply(lambda r: r["DefenseFinder"] / genome_counts_map[r["group"]], axis=1)
combined["PADLOC_norm"] = combined.apply(lambda r: r["PADLOC"] / genome_counts_map[r["group"]], axis=1)

width = 0.35
x = range(len(combined))
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar([i - width/2 for i in x], combined["DefenseFinder_norm"], width,
       label="DefenseFinder", color=tool_colors["DefenseFinder"], edgecolor="#888888", linewidth=0.6)
ax.bar([i + width/2 for i in x], combined["PADLOC_norm"], width,
       label="PADLOC", color=tool_colors["PADLOC"], edgecolor="#888888", linewidth=0.6)
ax.set_xticks(list(x))
ax.set_xticklabels(combined["group"], rotation=15)
ax.set_ylabel("Avg. Defense Systems per Genome")
ax.set_title("DefenseFinder vs PADLOC — Systems per Genome by Group", fontsize=12, fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/home/ottena/mags_project/defense_comparison_fig1_bar.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Plot 2: Scatter ---
df_per_genome = df_defense.groupby(["plot_label", "group"]).size().reset_index(name="DF_count")
pl_per_genome = df_padloc.groupby(["plot_label", "group"]).size().reset_index(name="PL_count")
scatter_df = df_per_genome.merge(pl_per_genome, on=["plot_label", "group"], how="outer").fillna(0)
x_all = scatter_df["DF_count"].values
y_all = scatter_df["PL_count"].values

fig, ax = plt.subplots(figsize=(8, 7))
for group, gdf in scatter_df.groupby("group"):
    ax.scatter(gdf["DF_count"], gdf["PL_count"], label=group,
               color=group_colors[group], s=100, edgecolor="#888888", linewidth=0.6, zorder=3)
b, m = polyfit(x_all, y_all, 1)
x_line = sorted(x_all)
ax.plot(x_line, [m*xi + b for xi in x_line], color="#AAAAAA", linestyle="--", linewidth=1.2)
ax.set_xlabel("DefenseFinder — Systems per Genome")
ax.set_ylabel("PADLOC — Systems per Genome")
ax.set_title("Per-Genome System Counts: DefenseFinder vs PADLOC", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig("/home/ottena/mags_project/defense_comparison_fig2_scatter.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Plot 3: Functional proportions ---
df_func = df_defense["functional_category"].value_counts(normalize=True).reset_index()
df_func.columns = ["functional_category", "DefenseFinder"]
pl_func = df_padloc["functional_category"].value_counts(normalize=True).reset_index()
pl_func.columns = ["functional_category", "PADLOC"]
func_comp = df_func.merge(pl_func, on="functional_category").sort_values("DefenseFinder", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
x = range(len(func_comp))
ax.barh([i - width/2 for i in x], func_comp["DefenseFinder"], width,
        label="DefenseFinder", color=tool_colors["DefenseFinder"], edgecolor="#888888", linewidth=0.6)
ax.barh([i + width/2 for i in x], func_comp["PADLOC"], width,
        label="PADLOC", color=tool_colors["PADLOC"], edgecolor="#888888", linewidth=0.6)
ax.set_yticks(list(x))
ax.set_yticklabels(func_comp["functional_category"])
ax.set_xlabel("Proportion of Total Systems")
ax.set_title("Functional Category Proportions: DefenseFinder vs PADLOC", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig("/home/ottena/mags_project/defense_comparison_fig3_functional.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Plot 4: Venn ---
df_system_types = set(df_defense["type"].str.lower().unique())
pl_system_types = set(df_padloc["system"].str.lower().unique())
fig, ax = plt.subplots(figsize=(8, 6))
venn2([df_system_types, pl_system_types], set_labels=("DefenseFinder", "PADLOC"),
      set_colors=(tool_colors["DefenseFinder"], tool_colors["PADLOC"]), alpha=0.8, ax=ax)
ax.set_title("Overlap of Defense System Types Detected", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("/home/ottena/mags_project/defense_comparison_fig4_venn.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Plot 5: Functional breakdown by group per tool ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
for ax, (tool_df, tool_name) in zip(axes, [(df_defense, "DefenseFinder"), (df_padloc, "PADLOC")]):
    pivot = tool_df.groupby(["group", "functional_category"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(groups_list)
    pivot = pivot.reindex(columns=[c for c in func_cats if c in pivot.columns], fill_value=0)
    for g in groups_list:
        pivot.loc[g] = pivot.loc[g] / genome_counts_map[g]
    bottom = pd.Series([0.0] * len(pivot), index=pivot.index)
    for cat in pivot.columns:
        ax.bar(range(len(pivot)), pivot[cat], bottom=bottom,
               label=cat, color=func_colors[cat], edgecolor="#888888", linewidth=0.4, width=0.5)
        bottom += pivot[cat]
    ax.set_xticks(range(len(groups_list)))
    ax.set_xticklabels(groups_list, rotation=15, fontsize=10)
    ax.set_title(tool_name, fontsize=13, fontweight="bold")
    ax.set_ylabel("Avg. Defense Systems per Genome")
    ax.spines[["top", "right"]].set_visible(False)

axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, title="Functional Category")
fig.suptitle("Functional Category Breakdown by Group: DefenseFinder vs PADLOC",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/home/ottena/mags_project/defense_comparison_fig5_group_func.png", dpi=300, bbox_inches="tight")
plt.show()

print("All 5 figures saved!")

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/35.3 MB 2.7 MB/s eta 0:00:11